# S00：YOLO11n 正式实验

这是用于 Ocean Engineering 论文的独立正式实验 Notebook。

- 论文别名：`S0`
- 模型 YAML：`experiments/formal_models/R00_yolo11n_baseline.yaml`
- 检测步长：`[8, 16, 32]`
- 随机种子由正式实验注册表固定为 `0`，不会循环运行多个随机种子。
- 运行到训练步骤时，全部检查通过后会直接开始训练，无需修改任何开关。
- 本实验使用冻结的 HRSC2016-MS YOLO-HBB 数据集；训练/验证/测试集固定为 610/460/610 幅图像，测试集不参与模型选择。


## 1. 挂载云盘、安装固定环境并获取实验代码

下面的单元格一次完成 Google Drive 挂载、Ultralytics 8.4.92 安装、私有仓库认证、固定提交检出和运行环境核验。它只读取 Colab Secret 中已有的 `GITHUB_TOKEN`，不会把令牌写入 URL、Notebook 或仓库。若认证失败，程序会立即停止，不会继续执行训练。


In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

import base64
import os
import platform
import subprocess
import sys
from pathlib import Path

# 本实验的编号和代码版本已经固定，无需手工修改。
RUN_ID = "S00"
FORMAL_CODE_COMMIT = "39e060dd074335dcb929c59a1ffdc4e1a5186ff6"
REPOSITORY_URL = "https://github.com/HoverdZ/ship-yolo.git"
REPOSITORY_DIR = "/content/ship-yolo"

# 安装此前 6 个 InceptionDW 正式实验使用的 Ultralytics 版本。
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "ultralytics==8.4.92"],
    check=True,
)

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception as error:
    raise RuntimeError(
        "无法读取 Colab Secret：GITHUB_TOKEN。请先完成 GitHub 身份认证，"
        "然后从本单元格重新运行；当前程序已停止。"
    ) from error
if not github_token:
    raise RuntimeError(
        "Colab Secret 中没有可用的 GITHUB_TOKEN。请添加该 Secret 并允许"
        "当前 Notebook 访问；当前程序已停止。"
    )


def git_run(arguments, cwd=None):
    basic = base64.b64encode(
        f"x-access-token:{github_token}".encode("utf-8")
    ).decode("ascii")
    command = [
        "git",
        "-c",
        f"http.extraHeader=AUTHORIZATION: basic {basic}",
        *list(arguments),
    ]
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        stderr = result.stderr.replace(github_token, "***")
        raise RuntimeError(
            "Git 操作失败。请先处理 GitHub 身份认证或仓库状态，"
            "不要继续运行后续单元格。\n" + stderr[-2000:]
        )
    return result.stdout.strip()


repo = Path(REPOSITORY_DIR)
if repo.exists():
    if not (repo / ".git").is_dir():
        raise FileExistsError(
            f"{repo} 已存在，但不是预期的 Git 仓库。程序没有删除该目录。"
        )
    dirty = git_run(
        ["status", "--porcelain", "--untracked-files=no"],
        cwd=repo,
    )
    if dirty:
        raise RuntimeError("现有仓库包含未提交的受跟踪文件修改，已拒绝切换版本。")
else:
    git_run(
        [
            "clone",
            "--filter=blob:none",
            "--no-checkout",
            REPOSITORY_URL,
            str(repo),
        ]
    )

git_run(["fetch", "--depth=1", "origin", FORMAL_CODE_COMMIT], cwd=repo)
git_run(["checkout", "--detach", FORMAL_CODE_COMMIT], cwd=repo)
actual_commit = git_run(["rev-parse", "HEAD"], cwd=repo)
assert actual_commit == FORMAL_CODE_COMMIT, (
    actual_commit,
    FORMAL_CODE_COMMIT,
)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("固定仓库提交：", actual_commit)

import torch
import ultralytics

assert ultralytics.__version__ == "8.4.92", ultralytics.__version__
print("Python 版本：", platform.python_version())
print("PyTorch 版本：", torch.__version__)
print("CUDA 版本：", torch.version.cuda)
print("cuDNN 版本：", torch.backends.cudnn.version())
print("Ultralytics 版本：", ultralytics.__version__)
print(
    "GPU：",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
)

DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ship_detection/paper_project"
)
for relative in (
    "datasets",
    "repository_snapshots",
    "formal_experiments",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
    "paper_artifacts/visualizations",
    "paper_artifacts/manifests",
    "exports",
):
    (DRIVE_PROJECT_ROOT / relative).mkdir(parents=True, exist_ok=True)
print("论文项目云盘目录：", DRIVE_PROJECT_ROOT)


## 2. 准备 HRSC2016-MS、完成训练前审计并直接开始正式训练

下面的单元格读取云盘 `ship_detection/data_2/HRSC2016_MS_YOLO.zip`：先把单个 ZIP 高速复制到 Colab 本地并实时显示字节进度，再安全解压并同时显示文件数和字节进度。程序不会重新划分数据，而是固定使用归档中的 train/val/test = 610/460/610；随后重建本地 `data.yaml`，检查 1680 幅图像、7655 个船舶实例、图片与标签对应关系、标签合法性、跨划分重复、模型结构、CPU 前向/反向、复杂度和官方预训练权重继承。全部检查通过后，在当前内核中直接调用官方 `YOLO.train(...)`，完整 epoch 输出实时显示；存在有效 `last.pt` 时自动续训。


In [ ]:
from dataclasses import replace
from pathlib import Path
import shutil

from tools.formal_experiments.hrsc2016_ms import prepare_hrsc2016_ms_archive
from tools.formal_experiments.protocol import (
    FormalRunConfig,
    prepare_experiment,
    print_run_banner,
    resolve_run_state,
    train_foreground,
)

# 路径和正式划分已经固定，无需修改任何开关。
HRSC_ARCHIVE = Path(
    "/content/drive/MyDrive/ship_detection/data_2/HRSC2016_MS_YOLO.zip"
)
dataset = prepare_hrsc2016_ms_archive(
    HRSC_ARCHIVE,
    local_archive_path="/content/dataset_archives/HRSC2016_MS_YOLO.zip",
    extract_root="/content/ship_detection/HRSC2016_MS_YOLO",
    runtime_yaml="/content/ship_detection/hrsc2016_ms_runtime.yaml",
    descriptor_path="/content/ship_detection/hrsc2016_ms_descriptor.yaml",
    audit_output="/content/ship_detection/hrsc2016_ms_integration_audit.json",
    artifact_dir=DRIVE_PROJECT_ROOT / "datasets" / "HRSC2016-MS",
    show_progress=True,
)

# 数据已经由上面的步骤准备到 Colab 本地。将源根和目标根设为同一路径，
# 正式协议只做只读复核，不会再复制第二份 2.5 GB 数据。
config = FormalRunConfig.from_registry(
    RUN_ID,
    run_training=True,
    data_yaml_override=str(dataset["data_yaml"]),
    drive_data_root_override=str(dataset["root"]),
)
config = replace(
    config,
    local_data_root=str(dataset["root"]),
    local_yaml=Path(dataset["data_yaml"]),
)
config.protocol_staging_dir.mkdir(parents=True, exist_ok=True)
for source in (dataset["descriptor"], dataset["audit"], dataset["manifest"]):
    shutil.copyfile(source, config.protocol_staging_dir / Path(source).name)

run_mode = resolve_run_state(config)
print("运行方式：", "从断点续训" if run_mode == "resume" else "全新训练")

prepared = prepare_experiment(config)
assert prepared["structure"]["passed"], prepared["structure"]
print("数据集划分审计：", prepared["dataset_audit"]["splits"])
print("预训练权重 Loaded/Total：", prepared["transfer"]["loaded_total"])
print("结构审计：通过")
print("模型规模：", prepared["model_info"])

print_run_banner(config)
print("全部训练前检查通过，开始正式训练。")
trained_model, train_results, drive_mirror = train_foreground(
    config,
    initialized_model=prepared["model"],
)

# 后处理会从 best.pt 单独加载模型，因此先释放训练器、优化器和旧模型显存。
import contextlib
import gc

prepared_model = prepared.pop("model", None)
del prepared_model, trained_model, train_results
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    with contextlib.suppress(Exception):
        torch.cuda.ipc_collect()
    print(
        "训练对象已释放，当前显存：",
        f"allocated={torch.cuda.memory_allocated() / 1024**3:.2f} GiB，",
        f"reserved={torch.cuda.memory_reserved() / 1024**3:.2f} GiB",
    )


## 3. 完成最终验证、更新论文结果表并核验备份

训练结束后运行下面的单元格。训练器显存已经在上一单元格末尾释放；本单元格使用 `best.pt` 完成固定验证，并把逐图统计限制在不超过 8 张的小批次，避免将整个验证集一次送入显存。所有状态文件稳定后才生成校验清单和 ZIP，随后原子同步到 Google Drive。测试集仍保持封存，不参与模型选择。


In [ ]:
from tools.formal_experiments.protocol import finalize_run
from tools.paper_artifacts.results.builders import TABLES, build
from tools.windows_collection import verify_checksum_manifest

manifest = finalize_run(config, mirror=drive_mirror)
print("最终验证指标：", manifest["validation_metrics"])
print("已完成的云盘实验目录：", config.drive_dir)

table_root = DRIVE_PROJECT_ROOT / "paper_artifacts" / "tables"
run_root = DRIVE_PROJECT_ROOT / "formal_experiments"
for table_name in TABLES:
    paths = build(table_name, run_root, table_root / table_name)
    print("已更新结果表：", table_name, paths)

checksum_file = config.run_dir / "artifact_checksums.sha256"
assert checksum_file.is_file(), checksum_file
checks = verify_checksum_manifest(checksum_file)
failures = [row for row in checks if not row["passed"]]
assert not failures, failures[:10]
assert not (config.run_dir / "RUNNING.lock").exists()
assert not (config.drive_dir / "RUNNING.lock").exists()

export_zip = (
    DRIVE_PROJECT_ROOT
    / "exports"
    / f"{config.run_id}_{config.run_name}.zip"
)
assert export_zip.is_file(), export_zip
print(f"已通过 {len(checks)} 项本地文件校验。")
print("ZIP 备份：", export_zip)
print("运行清单：", config.drive_dir / "run_manifest.json")
